In [2]:
import math
from typing import List, Tuple
import torch
import torch.nn as nn

# 1. LoRA 레이어 정의 (Bias 버그 수정 완료)
class LinearLoRA(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, r: int = 8, lora_alpha: int = 16, lora_dropout: float = 0., bias: bool = True):
        super().__init__()
        self.r = r
        self.lora_alpha = lora_alpha
        self.lora_dropout = nn.Dropout(lora_dropout)
        assert r > 0, "Rank should be > 0."

        # 기존 가중치 동결
        self.pretrained = nn.Linear(in_dim, out_dim, bias=bias)
        self.pretrained.weight.requires_grad = False
        if bias:
            self.pretrained.bias.requires_grad = False

        # LoRA A, B 행렬 초기화
        self.lora_A = nn.Linear(in_dim, r, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))

        self.lora_B = nn.Linear(r, out_dim, bias=False)
        nn.init.constant_(self.lora_B.weight, 0)

        self.scaling = self.lora_alpha / self.r

    def forward(self, x):
        pretrained_out = self.pretrained(x)
        lora_out = self.lora_dropout(x)
        lora_out = self.lora_A(lora_out)
        lora_out = self.lora_B(lora_out)
        lora_out = lora_out * self.scaling
        return pretrained_out + lora_out


# 2. 헬퍼 함수: 기존 Linear를 LinearLoRA로 변환
def create_lora(module, r, lora_dropout, lora_alpha):
    k, d = module.weight.shape
    has_bias = module.bias is not None
    lora = LinearLoRA(d, k, r, lora_dropout=lora_dropout, lora_alpha=lora_alpha, bias=has_bias)

    with torch.no_grad():
        lora.pretrained.weight.copy_(module.weight)
        if has_bias:
            lora.pretrained.bias.copy_(module.bias)
    return lora


# 3. 핵심 함수: 모델을 순회하며 Q, K, V 레이어에 LoRA 주입 (재귀 버그 수정 완료)
def add_lora_layers(
    model,
    # 허깅페이스 모델들의 범용적인 Q, K, V 이름을 모두 포함
    module_names: Tuple = ("query", "key", "value", "q_proj", "k_proj", "v_proj"),
    r: int = 8,
    lora_alpha: float = 16,
    lora_dropout: float = 0.1,
    ignore_layers: List[str] = None
):
    if ignore_layers is None:
        ignore_layers = []

    # Dropout 비활성화 (기존 로직 유지)
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.p = 0.0

    # dict로 감싸서 순회 중 사이즈 변경 에러 방지
    for name, module in dict(model.named_modules()).items():
        if isinstance(module, nn.Linear) and any(mod_name in name for mod_name in module_names):
            if any(ign in name for ign in ignore_layers):
                continue

            print(f"✅ LoRA 주입 성공: {name}")
            temp_lora = create_lora(module, r=r, lora_dropout=lora_dropout, lora_alpha=lora_alpha)

            parent_name, child_name = name.rsplit('.', 1) if '.' in name else ('', name)

            if parent_name == '':
                setattr(model, child_name, temp_lora)
            else:
                parent_module = model.get_submodule(parent_name)
                setattr(parent_module, child_name, temp_lora)


# 4. 헬퍼 함수: LoRA가 적용된 파라미터만 학습 가능하도록 설정
def unfreeze_lora_only(model):
    # 전체 동결
    for param in model.parameters():
        param.requires_grad = False
    # lora_A, lora_B만 학습 활성화
    for name, param in model.named_parameters():
        if "lora_A" in name or "lora_B" in name:
            param.requires_grad = True


# ==========================================
# 🚀 [테스트] 실제 모델에 적용해보기
# ==========================================
if __name__ == "__main__":
    # 가상의 트랜스포머 Attention 블록 생성
    class DummyAttention(nn.Module):
        def __init__(self, hidden_size):
            super().__init__()
            self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
            self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
            self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
            self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

        def forward(self, x):
            # 복잡한 계산은 생략
            pass

    # 모델 초기화
    model = DummyAttention(hidden_size=768)

    print("--- [LoRA 주입 전] 모델 구조 ---")
    print(model)
    print("\n")

    # Q, K, V 모듈에 LoRA 주입
    add_lora_layers(
        model=model,
        module_names=("q_proj", "k_proj", "v_proj"), # Q, K, V 타겟팅
        r=8,
        lora_alpha=16
    )

    # 학습을 위해 LoRA 파라미터만 unfreeze
    unfreeze_lora_only(model)

    print("\n--- [LoRA 주입 후] 모델 구조 ---")
    print(model)

--- [LoRA 주입 전] 모델 구조 ---
DummyAttention(
  (q_proj): Linear(in_features=768, out_features=768, bias=False)
  (k_proj): Linear(in_features=768, out_features=768, bias=False)
  (v_proj): Linear(in_features=768, out_features=768, bias=False)
  (o_proj): Linear(in_features=768, out_features=768, bias=False)
)


✅ LoRA 주입 성공: q_proj
✅ LoRA 주입 성공: k_proj
✅ LoRA 주입 성공: v_proj

--- [LoRA 주입 후] 모델 구조 ---
DummyAttention(
  (q_proj): LinearLoRA(
    (lora_dropout): Dropout(p=0.1, inplace=False)
    (pretrained): Linear(in_features=768, out_features=768, bias=False)
    (lora_A): Linear(in_features=768, out_features=8, bias=False)
    (lora_B): Linear(in_features=8, out_features=768, bias=False)
  )
  (k_proj): LinearLoRA(
    (lora_dropout): Dropout(p=0.1, inplace=False)
    (pretrained): Linear(in_features=768, out_features=768, bias=False)
    (lora_A): Linear(in_features=768, out_features=8, bias=False)
    (lora_B): Linear(in_features=8, out_features=768, bias=False)
  )
  (v_proj): LinearLo